In [1]:
# ==========================================================
# Wine Quality - TPOT (CLASSROOM SAFE VERSION)
# ==========================================================

import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

from ydata_profiling import ProfileReport
import kagglehub
import joblib

from tpot import TPOTClassifier

# =========================
# CONFIGURAÇÕES DIDÁTICAS
# =========================
RANDOM_STATE = 42
TEST_SIZE = 0.2

GENERATIONS = 5        # poucos para aula
POPULATION_SIZE = 20  # controla custo computacional
MAX_TIME_MINS = 3     # limite total (ideal em sala)

BASE_DIR = "wine_tpot"
DIR_REPORTS = os.path.join(BASE_DIR, "reports")
DIR_FIGURES = os.path.join(BASE_DIR, "figures")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)
os.makedirs(DIR_FIGURES, exist_ok=True)

print("✅ TPOT – execução segura para sala de aula")

# =========================
# FUNÇÕES AUXILIARES
# =========================
def load_dataset():
    path = kagglehub.dataset_download("yasserh/wine-quality-dataset")
    csv = [c for c in glob.glob(os.path.join(path, "*.csv")) if "WineQT" in c][0]
    return pd.read_csv(csv).drop(columns=["Id"])


def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


def feature_engineering(df):
    eps = 1e-9
    df["total_acidity"] = df["fixed_acidity"] + df["volatile_acidity"] + df["citric_acid"]
    df["alcohol_sugar_ratio"] = df["alcohol"] / (df["residual_sugar"] + eps)
    df["so2_ratio"] = df["free_sulfur_dioxide"] / (df["total_sulfur_dioxide"] + eps)
    df["density_alcohol"] = df["density"] * df["alcohol"]
    return df


def plot_balance(y):
    plt.figure(figsize=(6,4))
    sns.countplot(x=y, color="maroon")
    plt.title("Distribuição das Classes")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_FIGURES, "class_balance.png"), dpi=200)
    plt.close()

# =========================
# MAIN
# =========================
def main():
    print("[1/5] Carregando dados...")
    df = load_dataset()
    df = normalize_columns(df)

    print("[2/5] EDA...")
    ProfileReport(df, title="Wine Quality - EDA (TPOT Classroom)") \
        .to_file(os.path.join(DIR_REPORTS, "eda_wine_quality.html"))

    plot_balance(df["quality"])

    print("[3/5] Feature Engineering...")
    df = feature_engineering(df)

    X = df.drop(columns=["quality"])

    le = LabelEncoder()
    y = le.fit_transform(df["quality"])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    print("[4/5] AutoML TPOT (Evolução Genética)...")


    
    tpot = TPOTClassifier(
        generations=GENERATIONS,
        population_size=POPULATION_SIZE,
        scorers=["f1_macro"],
        scorers_weights=[1.0],
        cv=5,
        random_state=RANDOM_STATE,
        max_time_mins=MAX_TIME_MINS,
        n_jobs=1,      # ✅ evita Dask (principal correção)
        verbose=2
    )


    

    tpot.fit(X_train, y_train)

    print("\n✅ Pipeline final encontrado pelo TPOT:")
    print(tpot.fitted_pipeline_)

    y_pred = tpot.predict(X_test)
    f1 = f1_score(y_test, y_pred, average="macro")

    print("\n✅ F1-macro:", round(f1, 4))
    print(classification_report(y_test, y_pred))

    # =========================
    # SALVAMENTO
    # =========================
    joblib.dump(
        tpot.fitted_pipeline_,
        os.path.join(BASE_DIR, "model_pipeline_tpot.joblib")
    )

    meta = {
        "framework": "tpot",
        "task": "classification",
        "target": "quality",
        "original_classes": le.classes_.tolist(),
        "features": X.columns.tolist(),
        "generations": GENERATIONS,
        "population_size": POPULATION_SIZE
    }

    with open(os.path.join(BASE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print("\n✅ TPOT FINALIZADO COM SUCESSO")
    print("➡ Pode executar novamente em aula sem medo")

# =========================
if __name__ == "__main__":
    main()

✅ TPOT – execução segura para sala de aula
[1/5] Carregando dados...
[2/5] EDA...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

[3/5] Feature Engineering...
[4/5] AutoML TPOT (Evolução Genética)...


Generation:  60%|████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                | 3/5 [03:04<02:02, 61.41s/it]



✅ Pipeline final encontrado pelo TPOT:
Pipeline(steps=[('normalizer', Normalizer(norm='l1')),
                ('passthrough', Passthrough()),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('featureunion-2',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('randomforestclassifier',
                 RandomForestClassifier(bootstrap=False,
                                        class_weight='balanced',
                                        max_features=0.0149303572275,
              